# Main code

In [ ]:
# =========================================================
# Ablation Study: ITE Interval Source Ablation
# No conformalized extension
#
# Compared variants:
#   B0: Point only
#   B1: Encoder MC latent interval
#   B2: Latent-GP smoothed MC interval
#   B3: FullGP GP-ITE posterior
#
# Experimental setting kept consistent with the main IntervalGP-VAE code:
#   - n_train = 1000
#   - n_test  = 50
#   - n_samples = 100
#   - 24 synthetic combinations
#   - latent_dim = 1
#   - hidden_dim = 64
#   - GP_LENGTHSCALE = 7
#   - GP_VARIANCE = 2.0
#   - GP_NOISE = 1e-4
#   - three-stage training
#   - AdamW optimizer
# =========================================================

import os
import time
import math
import random
import inspect
import logging
import itertools
import textwrap
from copy import deepcopy
from statistics import NormalDist

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch.utils.data import TensorDataset, DataLoader


# =========================================================
# 0. Basic settings
# =========================================================

logging.basicConfig(level=logging.INFO)

torch.set_default_tensor_type("torch.FloatTensor")

OUTPUT_DIR = "ablation_intervalgpvae_no_conformal"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = "cpu"
GLOBAL_SEED = 420

np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
random.seed(GLOBAL_SEED)

# For debugging, set MAX_ITERATIONS = 2.
# For all 24 combinations, set MAX_ITERATIONS = None.
MAX_ITERATIONS = None

chosen_version = "u_aux"

# =========================================================
# Main IntervalGP-VAE hyperparameters
# Keep consistent with the main IntervalGP-VAE experiment
# =========================================================

INTERVALGPVAE_LATENT_DIM = 1
INTERVALGPVAE_HIDDEN_DIM = 64
CAUSAL_HEAD_HIDDEN_DIM = 64

GP_LENGTHSCALE = 7
GP_VARIANCE = 2.0
GP_NOISE = 1e-4

BATCH_SIZE = 128

JOINT_EPOCHS = 200
HEAD_EPOCHS = 100
VAE_REFINE_EPOCHS = 50

LR_JOINT = 1e-3
LR_HEAD = 1e-3
LR_VAE_REFINE = 1e-4
WEIGHT_DECAY = 1e-5

ITE_CI = 0.90
Z_VALUE_90 = 1.6448536269514722

N_TRAIN = 1000
N_TEST = 50
N_SAMPLES_ITE = 100

# The main paper/table focuses on 90% intervals.
# You can add more levels if needed, e.g. (0.50, 0.60, 0.70, 0.80, 0.90, 0.95)
NOMINAL_LEVELS = (0.90,)

# If GP is too slow, set MAX_GP_POINTS = 300.
# To keep exact FullGP as in the main experiment, keep it as None.
MAX_GP_POINTS = None


# =========================================================
# 1. Utility functions
# =========================================================

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)


def z_value_from_nominal(nominal):
    return NormalDist().inv_cdf(0.5 + nominal / 2.0)


def to_numpy_1d(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    return np.asarray(x).reshape(-1)


def safe_corr(x, y):
    x = to_numpy_1d(x)
    y = to_numpy_1d(y)

    valid = np.isfinite(x) & np.isfinite(y)

    if valid.sum() < 2:
        return np.nan

    if np.std(x[valid]) == 0 or np.std(y[valid]) == 0:
        return np.nan

    return float(np.corrcoef(x[valid], y[valid])[0, 1])


def pehe_np(est_ite, true_ite):
    est_ite = to_numpy_1d(est_ite)
    true_ite = to_numpy_1d(true_ite)

    valid = np.isfinite(est_ite) & np.isfinite(true_ite)

    if valid.sum() == 0:
        return np.nan

    return float(np.sqrt(np.mean((est_ite[valid] - true_ite[valid]) ** 2)))


def print_function_source(title, funcs):
    print(f"\n{title}:")

    if isinstance(funcs, list):
        for i, f in enumerate(funcs):
            print(f"  Function {i + 1}:")
            try:
                src = inspect.getsource(f).strip()
                print(textwrap.indent(src, "    "))
            except Exception:
                print("    [source not available]")
    else:
        try:
            src = inspect.getsource(funcs).strip()
            print(textwrap.indent(src, "  "))
        except Exception:
            print("  [source not available]")


def maybe_subsample_gp_reference(x, *ys, max_points=None, seed=0):
    """
    Subsample GP reference points if max_points is not None.
    This still uses full GP posterior over the selected reference set.
    """
    n = x.shape[0]

    if max_points is None or n <= max_points:
        return (x, *ys)

    rng = np.random.default_rng(seed)
    idx = rng.choice(np.arange(n), size=max_points, replace=False)
    idx = np.sort(idx)

    idx_t = torch.tensor(idx, dtype=torch.long, device=x.device)

    out = [x[idx_t]]

    for y in ys:
        if torch.is_tensor(y):
            out.append(y[idx_t])
        else:
            out.append(np.asarray(y)[idx])

    return tuple(out)


def standardize_train_test_z(z_train, z_test):
    """
    Standardize Z using training-set statistics.
    This is important because the GP lengthscale is defined in standardized Z-space.
    """
    z_mean = z_train.mean(dim=0, keepdim=True)
    z_std = z_train.std(dim=0, keepdim=True).clamp_min(1e-6)

    z_train_std = (z_train - z_mean) / z_std
    z_test_std = (z_test - z_mean) / z_std

    return z_train_std, z_test_std, z_mean, z_std


# =========================================================
# 2. Full GP posterior functions
# =========================================================

def rbf_kernel(x1, x2=None, lengthscale=1.0, variance=1.0):
    """
    RBF kernel over input vectors.
    """
    if x2 is None:
        x2 = x1

    if x1.ndim == 1:
        x1 = x1.view(-1, 1)

    if x2.ndim == 1:
        x2 = x2.view(-1, 1)

    dists = torch.cdist(x1, x2).pow(2)

    return variance * torch.exp(-0.5 * dists / (lengthscale ** 2))


def full_gp_posterior_point(
    x_train,
    y_train,
    x_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
    jitter=1e-5,
):
    """
    Standard full GP posterior for point-valued scalar observations.
    """
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + noise * torch.eye(n_train, device=x_train.device)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(n_train, device=x_train.device)
            )
            break
        except RuntimeError:
            current_jitter *= 10
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_point.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def full_gp_posterior_heteroscedastic(
    x_train,
    y_train,
    x_test,
    noise_var_train,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
    jitter=1e-5,
):
    """
    Full GP posterior with heteroscedastic diagonal observation noise.
    Used for ITE-space IntervalGP head.
    """
    x_train = x_train.float()
    x_test = x_test.float()
    y_train = y_train.float().view(-1, 1)

    noise_var_train = noise_var_train.float().view(-1)
    noise_var_train = torch.clamp(noise_var_train, min=min_noise)

    n_train = x_train.shape[0]
    n_test = x_test.shape[0]

    K = rbf_kernel(
        x_train,
        x_train,
        lengthscale=lengthscale,
        variance=variance,
    )

    K = K + torch.diag(noise_var_train)

    K_s = rbf_kernel(
        x_train,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = rbf_kernel(
        x_test,
        x_test,
        lengthscale=lengthscale,
        variance=variance,
    )

    K_ss = K_ss + jitter * torch.eye(n_test, device=x_test.device)

    current_jitter = jitter

    for _ in range(8):
        try:
            L = torch.linalg.cholesky(
                K + current_jitter * torch.eye(n_train, device=x_train.device)
            )
            break
        except RuntimeError:
            current_jitter *= 10
    else:
        raise RuntimeError("Cholesky failed in full_gp_posterior_heteroscedastic.")

    alpha = torch.cholesky_solve(y_train, L)

    posterior_mean = K_s.t() @ alpha

    v = torch.cholesky_solve(K_s, L)
    posterior_cov = K_ss - K_s.t() @ v

    posterior_cov = 0.5 * (posterior_cov + posterior_cov.t())

    return posterior_mean.squeeze(-1), posterior_cov


def latent_gp_posterior_smoothing(
    z_train,
    mu_u_train,
    z_test,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    noise=GP_NOISE,
):
    """
    Full GP posterior smoothing for latent confounder U.
    The training targets are encoder posterior means mu_u_train.
    Each latent dimension is smoothed independently.
    """
    if mu_u_train.ndim == 1:
        mu_u_train = mu_u_train.view(-1, 1)

    latent_dim = mu_u_train.shape[1]

    means = []
    vars_diag = []

    for r in range(latent_dim):
        mean_r, cov_r = full_gp_posterior_point(
            x_train=z_train,
            y_train=mu_u_train[:, r],
            x_test=z_test,
            lengthscale=lengthscale,
            variance=variance,
            noise=noise,
        )

        var_r = torch.diag(cov_r).clamp_min(1e-8)

        means.append(mean_r)
        vars_diag.append(var_r)

    mean = torch.stack(means, dim=1)
    std = torch.sqrt(torch.stack(vars_diag, dim=1))

    return mean, std


# =========================================================
# 3. Hidden-confounder recovery evaluation
# =========================================================

def align_recovered_u_to_true_u(u_recovered, u_true):
    """
    Latent variables are identifiable only up to sign and scale.
    We linearly align recovered u to true u:
        true_u ≈ a * recovered_u + b.
    """
    u_recovered = to_numpy_1d(u_recovered)
    u_true = to_numpy_1d(u_true)

    valid = np.isfinite(u_recovered) & np.isfinite(u_true)

    if valid.sum() < 2:
        return u_recovered, np.nan, np.nan, np.nan, np.nan

    x = u_recovered[valid]
    y = u_true[valid]

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    u_aligned = a * u_recovered + b

    corr = safe_corr(u_aligned, u_true)
    rmse = float(np.sqrt(np.mean((u_aligned[valid] - u_true[valid]) ** 2)))

    return u_aligned, float(a), float(b), corr, rmse


# =========================================================
# 4. IntervalGP-VAE model
# =========================================================

class DoubleEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.mean_u = nn.Linear(hidden_dim, latent_dim)
        self.logvar_u = nn.Linear(hidden_dim, latent_dim)

        self.mean_eps = nn.Linear(hidden_dim, input_dim)
        self.logvar_eps = nn.Linear(hidden_dim, input_dim)

        if self.use_aux:
            self.mean_ua0 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua0 = nn.Linear(hidden_dim, latent_dim)

            self.mean_ua1 = nn.Linear(hidden_dim, latent_dim)
            self.logvar_ua1 = nn.Linear(hidden_dim, latent_dim)

    def forward(self, z):
        h = F.relu(self.fc1(z))

        mu_u = self.mean_u(h)
        logvar_u = self.logvar_u(h)
        std_u = torch.exp(0.5 * logvar_u)

        mu_eps = self.mean_eps(h)
        logvar_eps = self.logvar_eps(h)
        std_eps = torch.exp(0.5 * logvar_eps)

        if self.use_aux:
            mu_ua0 = self.mean_ua0(h)
            logvar_ua0 = self.logvar_ua0(h)
            std_ua0 = torch.exp(0.5 * logvar_ua0)

            mu_ua1 = self.mean_ua1(h)
            logvar_ua1 = self.logvar_ua1(h)
            std_ua1 = torch.exp(0.5 * logvar_ua1)

            return (
                mu_u, std_u, logvar_u,
                mu_eps, std_eps, logvar_eps,
                mu_ua0, std_ua0, logvar_ua0,
                mu_ua1, std_ua1, logvar_ua1,
            )

        return mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps


class AdditiveDecoder(nn.Module):
    def __init__(
        self,
        latent_dim,
        hidden_dim,
        output_dim,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        factor = 1 + (2 if use_auxiliary_latents else 0)

        self.fc1 = nn.Linear(latent_dim * factor, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)

    def forward(self, u, eps, ua0=None, ua1=None):
        if self.use_aux:
            x = torch.cat([u, ua0, ua1], dim=1)
        else:
            x = u

        h = F.relu(self.fc1(x))
        return self.out(h) + eps


class GPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents

        self.encoder = DoubleEncoder(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.decoder = AdditiveDecoder(
            latent_dim=latent_dim,
            hidden_dim=hidden_dim,
            output_dim=input_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.gp_lengthscale = gp_lengthscale
        self.gp_variance = gp_variance
        self.gp_noise = gp_noise

    def reparameterize(self, mu, std):
        return mu + torch.randn_like(std) * std

    def local_interval_gp_regularizer(self, z, mu_u, std_u):
        """
        Local interval-valued GP prior regularizer.
        This matches the training-time interval GP regularization style
        used in the main IntervalGP-VAE implementation.
        """
        k_ii = self.gp_variance * torch.ones_like(mu_u)
        gp_std = torch.sqrt(k_ii + self.gp_noise)

        lower = mu_u - std_u
        upper = mu_u + std_u

        normal = torch.distributions.Normal(
            loc=torch.zeros_like(mu_u),
            scale=gp_std,
        )

        prob = normal.cdf(upper) - normal.cdf(lower)
        prob = torch.clamp(prob, min=1e-8)

        return -torch.sum(torch.log(prob))

    def get_latent_stats(self, z, var_name="u"):
        outputs = self.encoder(z)

        if var_name == "u":
            return outputs[0], outputs[1], outputs[2]

        if var_name == "eps":
            return outputs[3], outputs[4], outputs[5]

        if var_name == "ua0":
            if not self.use_aux:
                return None, None, None
            return outputs[6], outputs[7], outputs[8]

        if var_name == "ua1":
            if not self.use_aux:
                return None, None, None
            return outputs[9], outputs[10], outputs[11]

        raise ValueError(f"Invalid var_name: {var_name}")

    def sample_latent(self, z, var_name="u"):
        mu, std, _ = self.get_latent_stats(z, var_name=var_name)

        if mu is None:
            return None

        return self.reparameterize(mu, std)

    def forward(self, z):
        if self.use_aux:
            (
                mu_u, std_u, logvar_u,
                mu_eps, std_eps, logvar_eps,
                mu_ua0, std_ua0, logvar_ua0,
                mu_ua1, std_ua1, logvar_ua1,
            ) = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)
            ua0 = self.reparameterize(mu_ua0, std_ua0)
            ua1 = self.reparameterize(mu_ua1, std_ua1)

            z_recon = self.decoder(u, eps, ua0=ua0, ua1=ua1)

        else:
            mu_u, std_u, logvar_u, mu_eps, std_eps, logvar_eps = self.encoder(z)

            u = self.reparameterize(mu_u, std_u)
            eps = self.reparameterize(mu_eps, std_eps)

            ua0 = None
            ua1 = None

            z_recon = self.decoder(u, eps)

        recon_loss = F.mse_loss(z_recon, z, reduction="sum")

        # Keep the same weighting style as the main code
        kl_u = -0.001 * torch.sum(
            1 + logvar_u - mu_u.pow(2) - logvar_u.exp()
        )

        kl_eps = -0.5 * torch.sum(
            1 + logvar_eps - mu_eps.pow(2) - logvar_eps.exp()
        )

        gp_interval_reg = self.local_interval_gp_regularizer(
            z=z,
            mu_u=mu_u,
            std_u=std_u,
        )

        std_penalty = torch.sum(std_u ** 2)

        total_loss = (
            recon_loss
            + kl_u
            + kl_eps
            + gp_interval_reg
            + 10.0 * std_penalty
        )

        if self.use_aux:
            kl_ua0 = -0.5 * torch.sum(
                1 + logvar_ua0 - mu_ua0.pow(2) - logvar_ua0.exp()
            )

            kl_ua1 = -0.5 * torch.sum(
                1 + logvar_ua1 - mu_ua1.pow(2) - logvar_ua1.exp()
            )

            total_loss = total_loss + kl_ua0 + kl_ua1

        return total_loss, {
            "recon_loss": recon_loss.item(),
            "kl_u": kl_u.item(),
            "kl_eps": kl_eps.item(),
            "gp_interval_reg": gp_interval_reg.item(),
            "std_penalty": std_penalty.item(),
            "u": u,
            "eps": eps,
            "ua0": ua0,
            "ua1": ua1,
            "mu_u": mu_u,
            "std_u": std_u,
            "z_recon": z_recon,
        }


class FlexibleCausalHead(nn.Module):
    def __init__(
        self,
        latent_dim_u,
        hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
        z_y_dim=None,
        use_auxiliary_latents=False,
    ):
        super().__init__()

        self.use_aux = use_auxiliary_latents
        self.z_y_dim = z_y_dim

        input_dim = latent_dim_u + 1

        if z_y_dim is not None:
            input_dim += z_y_dim

        if use_auxiliary_latents:
            input_dim += latent_dim_u

        self.fc = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)

    def forward(self, u, t, z_y=None, ua1=None):
        t = t.view(-1, 1)

        parts = [u, t]

        if self.z_y_dim is not None:
            if z_y is None:
                raise ValueError("Expected z_y input but got None.")
            parts.append(z_y)

        if self.use_aux:
            if ua1 is None:
                raise ValueError("Expected ua1 input but got None.")
            parts.append(ua1)

        x = torch.cat(parts, dim=1)
        h = F.relu(self.fc(x))

        return self.out(h)


class CausalGPVAEwithNoise(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_dim,
        latent_dim,
        z_y_dim=None,
        use_auxiliary_latents=False,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ):
        super().__init__()

        self.z_y_dim = z_y_dim
        self.use_aux = use_auxiliary_latents

        self.vae = GPVAEwithNoise(
            input_dim=input_dim,
            hidden_dim=hidden_dim,
            latent_dim=latent_dim,
            gp_lengthscale=gp_lengthscale,
            gp_variance=gp_variance,
            gp_noise=gp_noise,
            use_auxiliary_latents=use_auxiliary_latents,
        )

        self.causal_head = FlexibleCausalHead(
            latent_dim_u=latent_dim,
            hidden_dim=CAUSAL_HEAD_HIDDEN_DIM,
            z_y_dim=z_y_dim,
            use_auxiliary_latents=use_auxiliary_latents,
        )

    def forward(self, z, t, y):
        loss_vae, vae_info = self.vae(z)

        u = vae_info["u"]
        ua1 = vae_info["ua1"] if self.use_aux else None

        z_y = make_z_y(self, z)

        y_pred = self.causal_head(
            u,
            t,
            z_y=z_y,
            ua1=ua1,
        )

        causal_loss = F.mse_loss(
            y_pred,
            y.unsqueeze(-1),
            reduction="sum",
        )

        total_loss = loss_vae + causal_loss

        return total_loss, {
            **vae_info,
            "y_pred": y_pred,
            "causal_loss": causal_loss.item(),
        }


def make_z_y(model, z):
    if model.causal_head.z_y_dim == z.shape[1]:
        return z

    if model.causal_head.z_y_dim:
        return z[:, z.shape[1] // 2:]

    return None


# =========================================================
# 5. Synthetic data
# =========================================================

proxy_func_sets = [
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.5 * u ** 2),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.exp(0.3 * u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 2,
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.exp(-0.2 * u ** 2),
        lambda u, ua0, ua1: torch.sin(2 * u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: u ** 3,
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
        lambda u, ua0, ua1: torch.sin(u),
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.tanh(0.5 * u),
        lambda u, ua0, ua1: torch.sin(u),
        lambda u, ua0, ua1: torch.exp(-0.3 * u ** 2),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: u ** 2,
    ],
    [
        lambda u, ua0, ua1: u,
        lambda u, ua0, ua1: torch.log1p(torch.exp(u)),
        lambda u, ua0, ua1: torch.exp(0.5 * u),
        lambda u, ua0, ua1: torch.tanh(u),
        lambda u, ua0, ua1: torch.sigmoid(u),
        lambda u, ua0, ua1: torch.log1p(u ** 2),
    ],
]


def treat_func_linear_1(u_np, ua0_np=None):
    logits = 0.8 * u_np
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


def treat_func_linear_2(u_np, ua0_np=None):
    logits = 0.4 * u_np + 0.3
    probs = 1.0 / (1.0 + np.exp(-logits))
    return np.random.binomial(1, probs).astype(np.float32).reshape(-1)


treatment_func_list = [
    treat_func_linear_1,
    treat_func_linear_2,
]


def outcome_func_1(u_np, t_np, ua0=None, ua1=None, noise_std=0.1):
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.2 * np.sin(u_np)
        + t_np.reshape(-1, 1) * (0.8 + 0.3 * u_np)
    )

    noise = np.random.normal(0.0, noise_std, size=base.shape)

    return (base + noise).astype(np.float32)


def outcome_func_2(u_np, t_np, ua0=None, ua1=None, noise_std=0.1):
    if t_np.ndim > 1:
        t_np = t_np.reshape(-1)

    base = (
        0.5 * u_np
        + 0.3 * np.cos(u_np)
        + t_np.reshape(-1, 1) * (0.5 + 0.5 * u_np)
    )

    noise = np.random.normal(0.0, noise_std, size=base.shape)

    return (base + noise).astype(np.float32)


outcome_func_list = [
    outcome_func_1,
    outcome_func_2,
]


def generate_synthetic_data_with_aux_uas(
    n=1000,
    noise_std=0.1,
    seed=0,
    num_proxies=None,
    proxy_funcs=None,
    outcome_func=None,
    treatment_func=None,
):
    if seed is not None:
        set_seed(seed)

    u_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua0_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)
    ua1_np = np.random.normal(0, 1, size=(n, 1)).astype(np.float32)

    u_tensor = torch.from_numpy(u_np)
    ua0_tensor = torch.from_numpy(ua0_np)
    ua1_tensor = torch.from_numpy(ua1_np)

    if proxy_funcs is None:
        proxy_funcs = proxy_func_sets[0]

    if num_proxies is None:
        num_proxies = len(proxy_funcs)

    proxy_funcs = proxy_funcs[:num_proxies]

    clean_z = torch.cat(
        [
            g(u_tensor, ua0_tensor, ua1_tensor)
            for g in proxy_funcs
        ],
        dim=1,
    )

    eps_z = torch.randn(n, num_proxies) * noise_std
    z_tensor = clean_z + eps_z

    if treatment_func is None:
        treatment_func = treat_func_linear_1

    t_np = treatment_func(u_np, ua0_np)
    t_tensor = torch.from_numpy(t_np.astype(np.float32))

    if outcome_func is None:
        outcome_func = outcome_func_1

    y_np = outcome_func(
        u_np,
        t_np,
        ua0_np,
        ua1_np,
        noise_std=noise_std,
    )

    y_tensor = torch.from_numpy(y_np.squeeze())

    return {
        "z": z_tensor.float(),
        "t": t_tensor.float(),
        "y": y_tensor.float(),
        "u": u_tensor.squeeze().float(),
        "ua0": ua0_tensor.squeeze().float(),
        "ua1": ua1_tensor.squeeze().float(),
        "proxy_funcs": proxy_funcs,
        "outcome_func": outcome_func,
        "treatment_func": treatment_func,
    }


def call_outcome_func(
    outcome_func,
    u_np,
    t_np,
    z_np=None,
    ua0_np=None,
    ua1_np=None,
    noise_std=0.0,
):
    """
    Use noise_std=0.0 for true structural ITE.
    """
    if torch.is_tensor(u_np):
        u_np = u_np.detach().cpu().numpy()

    if torch.is_tensor(t_np):
        t_np = t_np.detach().cpu().numpy()

    if ua0_np is not None and torch.is_tensor(ua0_np):
        ua0_np = ua0_np.detach().cpu().numpy()

    if ua1_np is not None and torch.is_tensor(ua1_np):
        ua1_np = ua1_np.detach().cpu().numpy()

    return outcome_func(
        u_np,
        t_np,
        ua0_np,
        ua1_np,
        noise_std=noise_std,
    )


def compute_true_ite_for_dataset(data, outcome_func):
    true_u = data["u"].detach().cpu().numpy().reshape(-1, 1)

    ua0 = data.get("ua0", None)
    ua1 = data.get("ua1", None)

    if ua0 is not None:
        ua0_np = ua0.detach().cpu().numpy().reshape(-1, 1)
    else:
        ua0_np = None

    if ua1 is not None:
        ua1_np = ua1.detach().cpu().numpy().reshape(-1, 1)
    else:
        ua1_np = None

    t0 = np.zeros_like(true_u)
    t1 = np.ones_like(true_u)

    y0 = call_outcome_func(
        outcome_func=outcome_func,
        u_np=true_u,
        t_np=t0,
        ua0_np=ua0_np,
        ua1_np=ua1_np,
        noise_std=0.0,
    )

    y1 = call_outcome_func(
        outcome_func=outcome_func,
        u_np=true_u,
        t_np=t1,
        ua0_np=ua0_np,
        ua1_np=ua1_np,
        noise_std=0.0,
    )

    return (y1 - y0).reshape(-1).astype(np.float32)


# =========================================================
# 6. Training function
# =========================================================

def build_model(z_train_all):
    use_aux = chosen_version == "u_aux"

    if chosen_version == "z_to_y":
        z_y_dim = z_train_all.shape[1]
    elif chosen_version == "split_z_to_t_and_y":
        z_y_dim = z_train_all.shape[1] // 2
    else:
        z_y_dim = None

    model = CausalGPVAEwithNoise(
        input_dim=z_train_all.shape[1],
        hidden_dim=INTERVALGPVAE_HIDDEN_DIM,
        latent_dim=INTERVALGPVAE_LATENT_DIM,
        z_y_dim=z_y_dim,
        use_auxiliary_latents=use_aux,
        gp_lengthscale=GP_LENGTHSCALE,
        gp_variance=GP_VARIANCE,
        gp_noise=GP_NOISE,
    ).to(DEVICE)

    return model


def train_intervalgpvae_model(
    model,
    z_train_all,
    t_train_all,
    y_train_all,
    u_train_true_all=None,
):
    if u_train_true_all is None:
        dataset = TensorDataset(
            z_train_all,
            t_train_all,
            y_train_all,
        )
    else:
        dataset = TensorDataset(
            z_train_all,
            t_train_all,
            y_train_all,
            u_train_true_all,
        )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    # -----------------------------------------------------
    # Stage 1: joint training with AdamW
    # -----------------------------------------------------
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR_JOINT,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(JOINT_EPOCHS):
        model.train()

        for batch in loader:
            z_batch = batch[0].to(DEVICE)
            t_batch = batch[1].to(DEVICE)
            y_batch = batch[2].to(DEVICE)

            loss, info = model(z_batch, t_batch, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # -----------------------------------------------------
    # Stage 2: train causal head with frozen VAE
    # -----------------------------------------------------
    for param in model.vae.parameters():
        param.requires_grad = False

    for param in model.causal_head.parameters():
        param.requires_grad = True

    causal_optimizer = torch.optim.AdamW(
        model.causal_head.parameters(),
        lr=LR_HEAD,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(HEAD_EPOCHS):
        model.train()

        for batch in loader:
            z_batch = batch[0].to(DEVICE)
            t_batch = batch[1].to(DEVICE)
            y_batch = batch[2].to(DEVICE)

            with torch.no_grad():
                mu_u = model.vae.get_latent_stats(
                    z_batch,
                    var_name="u",
                )[0]

                if model.use_aux:
                    mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
                        z_batch,
                        var_name="ua1",
                    )

                    ua1_batch = model.vae.reparameterize(
                        mu_ua1,
                        std_ua1,
                    )
                else:
                    ua1_batch = None

            z_y_batch = make_z_y(model, z_batch)

            y_pred = model.causal_head(
                mu_u.detach(),
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch.detach() if ua1_batch is not None else None,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
            )

            causal_optimizer.zero_grad()
            causal_loss.backward()
            causal_optimizer.step()

    # -----------------------------------------------------
    # Stage 3: refine VAE with frozen causal head
    # -----------------------------------------------------
    for param in model.causal_head.parameters():
        param.requires_grad = False

    for param in model.vae.parameters():
        param.requires_grad = True

    vae_optimizer = torch.optim.AdamW(
        model.vae.parameters(),
        lr=LR_VAE_REFINE,
        weight_decay=WEIGHT_DECAY,
    )

    for epoch in range(VAE_REFINE_EPOCHS):
        model.train()

        for batch in loader:
            z_batch = batch[0].to(DEVICE)
            t_batch = batch[1].to(DEVICE)
            y_batch = batch[2].to(DEVICE)

            z_y_batch = make_z_y(model, z_batch)

            loss_vae, vae_info = model.vae(z_batch)

            ua1_batch = (
                model.vae.sample_latent(z_batch, var_name="ua1")
                if model.use_aux
                else None
            )

            y_pred = model.causal_head(
                vae_info["u"],
                t_batch,
                z_y=z_y_batch,
                ua1=ua1_batch,
            )

            causal_loss = F.mse_loss(
                y_pred,
                y_batch.unsqueeze(-1),
                reduction="sum",
            )

            total_loss = loss_vae + causal_loss

            vae_optimizer.zero_grad()
            total_loss.backward()
            vae_optimizer.step()

    for param in model.parameters():
        param.requires_grad = True

    model.eval()

    return model


# =========================================================
# 7. ITE interval construction methods
# =========================================================

@torch.no_grad()
def point_ite_from_encoder_mean(model, z):
    """
    B0: Point-only ITE estimate.
    Lower and upper bounds are both set to this point estimate.
    """
    model.eval()

    mu_u, _, _ = model.vae.get_latent_stats(z, var_name="u")
    z_y = make_z_y(model, z)

    if model.use_aux:
        mu_ua1, _, _ = model.vae.get_latent_stats(
            z,
            var_name="ua1",
        )
        ua1 = mu_ua1
    else:
        ua1 = None

    t0 = torch.zeros(len(z), device=z.device)
    t1 = torch.ones(len(z), device=z.device)

    y0 = model.causal_head(
        mu_u,
        t0,
        z_y=z_y,
        ua1=ua1,
    ).squeeze(-1)

    y1 = model.causal_head(
        mu_u,
        t1,
        z_y=z_y,
        ua1=ua1,
    ).squeeze(-1)

    return y1 - y0


@torch.no_grad()
def sample_ite_from_encoder(
    model,
    z,
    n_samples=N_SAMPLES_ITE,
    nominal=ITE_CI,
):
    """
    B1: Direct MC sampling from encoder posterior q(u|z) and q(ua1|z).
    """
    model.eval()

    mu_u, std_u, _ = model.vae.get_latent_stats(z, var_name="u")

    N, d_u = mu_u.shape
    device = z.device

    z_y = make_z_y(model, z)

    if z_y is not None:
        ZY = (
            z_y.unsqueeze(0)
            .expand(n_samples, -1, -1)
            .reshape(n_samples * N, z_y.shape[1])
        )
    else:
        ZY = None

    eps_u = torch.randn(n_samples, N, d_u, device=device)
    u_samples = mu_u.unsqueeze(0) + eps_u * std_u.unsqueeze(0)

    U = u_samples.reshape(n_samples * N, d_u)

    if model.use_aux:
        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
            z,
            var_name="ua1",
        )

        d_ua1 = mu_ua1.shape[1]

        eps_ua1 = torch.randn(n_samples, N, d_ua1, device=device)
        ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)

        UA1 = ua1_samples.reshape(n_samples * N, d_ua1)
    else:
        UA1 = None

    t0 = torch.zeros(n_samples * N, device=device)
    t1 = torch.ones(n_samples * N, device=device)

    y0 = model.causal_head(
        U,
        t0,
        z_y=ZY,
        ua1=UA1,
    ).squeeze(-1)

    y1 = model.causal_head(
        U,
        t1,
        z_y=ZY,
        ua1=UA1,
    ).squeeze(-1)

    ite_samples = (y1 - y0).view(n_samples, N)

    alpha = 1.0 - nominal
    lower_q = alpha / 2.0
    upper_q = 1.0 - alpha / 2.0

    ite_mean = ite_samples.mean(dim=0)
    ite_std = ite_samples.std(dim=0)
    ite_lower = ite_samples.quantile(lower_q, dim=0)
    ite_upper = ite_samples.quantile(upper_q, dim=0)

    return {
        "samples": ite_samples,
        "mean": ite_mean,
        "std": ite_std,
        "lower": ite_lower,
        "upper": ite_upper,
    }


@torch.no_grad()
def sample_ite_from_latent_gp_smoothing(
    model,
    z_train,
    z_test,
    n_samples=N_SAMPLES_ITE,
    nominal=ITE_CI,
):
    """
    B2: Latent-GP smoothed MC interval.

    Difference from B1:
    - B1 samples U directly from the encoder posterior q(U|Z).
    - B2 first applies GP posterior smoothing to U over proxy space,
      then samples U from the GP-smoothed posterior.
    - It does not use the ITE-space FullGP posterior.
    """
    model.eval()

    mu_u_train, _, _ = model.vae.get_latent_stats(
        z_train,
        var_name="u",
    )

    u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
        z_train=z_train,
        mu_u_train=mu_u_train,
        z_test=z_test,
        lengthscale=GP_LENGTHSCALE,
        variance=GP_VARIANCE,
        noise=GP_NOISE,
    )

    N, d_u = u_test_smooth.shape
    device = z_test.device

    eps_u = torch.randn(n_samples, N, d_u, device=device)
    u_samples = u_test_smooth.unsqueeze(0) + eps_u * u_test_smooth_std.unsqueeze(0)

    U = u_samples.reshape(n_samples * N, d_u)

    z_y = make_z_y(model, z_test)

    if z_y is not None:
        ZY = (
            z_y.unsqueeze(0)
            .expand(n_samples, -1, -1)
            .reshape(n_samples * N, z_y.shape[1])
        )
    else:
        ZY = None

    if model.use_aux:
        mu_ua1, std_ua1, _ = model.vae.get_latent_stats(
            z_test,
            var_name="ua1",
        )

        d_ua1 = mu_ua1.shape[1]

        eps_ua1 = torch.randn(n_samples, N, d_ua1, device=device)
        ua1_samples = mu_ua1.unsqueeze(0) + eps_ua1 * std_ua1.unsqueeze(0)

        UA1 = ua1_samples.reshape(n_samples * N, d_ua1)
    else:
        UA1 = None

    t0 = torch.zeros(n_samples * N, device=device)
    t1 = torch.ones(n_samples * N, device=device)

    y0 = model.causal_head(
        U,
        t0,
        z_y=ZY,
        ua1=UA1,
    ).squeeze(-1)

    y1 = model.causal_head(
        U,
        t1,
        z_y=ZY,
        ua1=UA1,
    ).squeeze(-1)

    ite_samples = (y1 - y0).view(n_samples, N)

    alpha = 1.0 - nominal
    lower_q = alpha / 2.0
    upper_q = 1.0 - alpha / 2.0

    ite_mean = ite_samples.mean(dim=0)
    ite_std = ite_samples.std(dim=0)
    ite_lower = ite_samples.quantile(lower_q, dim=0)
    ite_upper = ite_samples.quantile(upper_q, dim=0)

    return {
        "samples": ite_samples,
        "mean": ite_mean,
        "std": ite_std,
        "lower": ite_lower,
        "upper": ite_upper,
        "u_test_smooth": u_test_smooth,
        "u_test_smooth_std": u_test_smooth_std,
    }


@torch.no_grad()
def ite_space_intervalgp_head(
    model,
    z_train,
    z_test,
    n_samples=N_SAMPLES_ITE,
    nominal=ITE_CI,
    lengthscale=GP_LENGTHSCALE,
    variance=GP_VARIANCE,
    min_noise=GP_NOISE,
):
    """
    B3: FullGP GP-ITE posterior interval.

    Steps:
      1. Use encoder posterior samples on training data to get interval-valued
         pseudo-observations of ITE:
             [tau_lower_train, tau_upper_train].
      2. Use latent GP posterior smoothing from Z to U for the test points.
      3. Use full GP posterior separately over lower, upper, and mean ITE functions.
      4. Add GP posterior uncertainty to the lower and upper bounds.
    """
    model.eval()

    # -----------------------------------------------------
    # Training pseudo-observations from encoder MC sampling
    # -----------------------------------------------------
    train_samples = sample_ite_from_encoder(
        model=model,
        z=z_train,
        n_samples=n_samples,
        nominal=nominal,
    )

    ite_mean_train = train_samples["mean"]
    ite_std_train = train_samples["std"]
    ite_lower_train = train_samples["lower"]
    ite_upper_train = train_samples["upper"]

    mu_u_train, _, _ = model.vae.get_latent_stats(
        z_train,
        var_name="u",
    )

    mu_u_test, _, _ = model.vae.get_latent_stats(
        z_test,
        var_name="u",
    )

    # -----------------------------------------------------
    # Latent GP posterior smoothing before ITE-space GP
    # -----------------------------------------------------
    u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
        z_train=z_train,
        mu_u_train=mu_u_train,
        z_test=z_test,
        lengthscale=lengthscale,
        variance=variance,
        noise=min_noise,
    )

    u_train_smooth = mu_u_train.detach()

    (
        u_train_gp,
        ite_lower_train_gp,
        ite_upper_train_gp,
        ite_mean_train_gp,
        ite_std_train_gp,
    ) = maybe_subsample_gp_reference(
        u_train_smooth,
        ite_lower_train.detach(),
        ite_upper_train.detach(),
        ite_mean_train.detach(),
        ite_std_train.detach(),
        max_points=MAX_GP_POINTS,
        seed=123,
    )

    noise_var_lower = ite_std_train_gp.pow(2) + min_noise
    noise_var_upper = ite_std_train_gp.pow(2) + min_noise
    noise_var_mean = ite_std_train_gp.pow(2) + min_noise

    # -----------------------------------------------------
    # ITE-space FullGP posterior
    # -----------------------------------------------------
    gp_lower_mean, gp_lower_cov = full_gp_posterior_heteroscedastic(
        x_train=u_train_gp,
        y_train=ite_lower_train_gp,
        x_test=u_test_smooth,
        noise_var_train=noise_var_lower,
        lengthscale=lengthscale,
        variance=variance,
        min_noise=min_noise,
    )

    gp_upper_mean, gp_upper_cov = full_gp_posterior_heteroscedastic(
        x_train=u_train_gp,
        y_train=ite_upper_train_gp,
        x_test=u_test_smooth,
        noise_var_train=noise_var_upper,
        lengthscale=lengthscale,
        variance=variance,
        min_noise=min_noise,
    )

    gp_ite_mean, gp_ite_cov = full_gp_posterior_heteroscedastic(
        x_train=u_train_gp,
        y_train=ite_mean_train_gp,
        x_test=u_test_smooth,
        noise_var_train=noise_var_mean,
        lengthscale=lengthscale,
        variance=variance,
        min_noise=min_noise,
    )

    gp_lower_std = torch.sqrt(torch.diag(gp_lower_cov).clamp_min(1e-8))
    gp_upper_std = torch.sqrt(torch.diag(gp_upper_cov).clamp_min(1e-8))
    gp_ite_std = torch.sqrt(torch.diag(gp_ite_cov).clamp_min(1e-8))

    # Ensure lower <= upper
    raw_lower = torch.minimum(gp_lower_mean, gp_upper_mean)
    raw_upper = torch.maximum(gp_lower_mean, gp_upper_mean)

    z_value = z_value_from_nominal(nominal)

    ite_lower_test = raw_lower - z_value * gp_lower_std
    ite_upper_test = raw_upper + z_value * gp_upper_std

    return {
        "mean": gp_ite_mean,
        "std": gp_ite_std,
        "lower": ite_lower_test,
        "upper": ite_upper_test,
        "u_test_smooth": u_test_smooth,
        "u_test_smooth_std": u_test_smooth_std,
        "u_test_raw": mu_u_test,
        "ite_lower_train": ite_lower_train,
        "ite_upper_train": ite_upper_train,
        "ite_mean_train": ite_mean_train,
        "ite_std_train": ite_std_train,
    }


# =========================================================
# 8. Metrics
# =========================================================

def compute_interval_score(
    lower,
    upper,
    true_value,
    nominal=0.90,
):
    alpha = 1.0 - nominal

    lower = torch.as_tensor(lower).detach().cpu().float().reshape(-1)
    upper = torch.as_tensor(upper).detach().cpu().float().reshape(-1)
    true_value = torch.as_tensor(true_value).detach().cpu().float().reshape(-1)

    width = upper - lower

    below = torch.clamp(lower - true_value, min=0.0)
    above = torch.clamp(true_value - upper, min=0.0)

    score = width + (2.0 / alpha) * below + (2.0 / alpha) * above

    return float(score.mean().item())


def compute_all_metrics(
    point,
    lower,
    upper,
    true_ite,
    nominal=0.90,
):
    point = torch.as_tensor(point).detach().cpu().float().reshape(-1)
    lower = torch.as_tensor(lower).detach().cpu().float().reshape(-1)
    upper = torch.as_tensor(upper).detach().cpu().float().reshape(-1)
    true_ite = torch.as_tensor(true_ite).detach().cpu().float().reshape(-1)

    pehe = torch.sqrt(
        F.mse_loss(point, true_ite)
    ).item()

    ate_error = torch.abs(
        point.mean() - true_ite.mean()
    ).item()

    covered = (
        (true_ite >= lower)
        & (true_ite <= upper)
    ).float()

    coverage = covered.mean().item()

    width = (upper - lower).mean().item()

    interval_score = compute_interval_score(
        lower=lower,
        upper=upper,
        true_value=true_ite,
        nominal=nominal,
    )

    return {
        "PEHE": pehe,
        "ATE_error": ate_error,
        "coverage": coverage,
        "mean_width": width,
        "interval_score": interval_score,
    }


# =========================================================
# 9. Plotting helpers
# =========================================================

def plot_metric_bars_at_90(summary_df, output_dir=OUTPUT_DIR):
    df90 = summary_df[summary_df["nominal"] == 0.90].copy()

    metrics = [
        "PEHE",
        "ATE_error",
        "coverage",
        "mean_width",
        "interval_score",
    ]

    paths = []

    for metric in metrics:
        plt.figure(figsize=(9, 4.8))

        plot_df = df90.copy()

        plt.bar(
            plot_df["method"],
            plot_df[metric],
        )

        plt.xticks(rotation=30, ha="right")
        plt.ylabel(metric)
        plt.title(f"{metric} at 90% nominal level")
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)
        plt.tight_layout()

        path = os.path.join(
            output_dir,
            f"ablation_no_conformal_{metric}_bar_90.png",
        )

        plt.savefig(path, dpi=300, bbox_inches="tight")
        plt.show()

        paths.append(path)

    return paths


def plot_coverage_width_tradeoff(summary_df, output_dir=OUTPUT_DIR):
    plt.figure(figsize=(7, 5))

    df90 = summary_df[summary_df["nominal"] == 0.90].copy()

    plt.scatter(
        df90["mean_width"],
        df90["coverage"],
        s=80,
    )

    for _, row in df90.iterrows():
        plt.text(
            row["mean_width"],
            row["coverage"],
            "  " + row["method"],
            fontsize=9,
            va="center",
        )

    plt.axhline(
        y=0.90,
        linestyle="--",
        linewidth=1.2,
        label="90% nominal level",
    )

    plt.xlabel("Mean interval width")
    plt.ylabel("Empirical coverage")
    plt.title("Coverage--width trade-off at 90% nominal level")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()

    path = os.path.join(
        output_dir,
        "ablation_no_conformal_coverage_width_tradeoff_90.png",
    )

    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()

    return path


# =========================================================
# 10. One-combination ablation experiment
# =========================================================

def run_one_combination_ablation(
    iteration_id,
    proxy_funcs,
    treat_func,
    outcome_func,
):
    print("\n" + "=" * 100)
    print(f"Iteration {iteration_id}")
    print("=" * 100)

    print_function_source("Proxy functions", proxy_funcs)
    print_function_source("Treatment function", treat_func)
    print_function_source("Outcome function", outcome_func)

    start_time = time.time()

    # -----------------------------------------------------
    # Generate data: same sample size and seed policy as main code
    # -----------------------------------------------------
    data_train = generate_synthetic_data_with_aux_uas(
        n=N_TRAIN,
        num_proxies=len(proxy_funcs),
        proxy_funcs=proxy_funcs,
        treatment_func=treat_func,
        outcome_func=outcome_func,
        seed=42,
    )

    data_test = generate_synthetic_data_with_aux_uas(
        n=N_TEST,
        num_proxies=len(proxy_funcs),
        proxy_funcs=proxy_funcs,
        treatment_func=treat_func,
        outcome_func=outcome_func,
        seed=100 + iteration_id,
    )

    z_train_all = data_train["z"].to(DEVICE)
    t_train_all = data_train["t"].to(DEVICE)
    y_train_all = data_train["y"].to(DEVICE)
    u_train_true_all = data_train["u"].to(DEVICE)

    z_test = data_test["z"].to(DEVICE)
    t_test = data_test["t"].to(DEVICE)
    y_test = data_test["y"].to(DEVICE)
    true_u_test = data_test["u"].to(DEVICE)

    # Standardize Z by training statistics
    z_train_all, z_test, z_mean, z_std = standardize_train_test_z(
        z_train_all,
        z_test,
    )

    true_ite_np = compute_true_ite_for_dataset(
        data=data_test,
        outcome_func=outcome_func,
    )

    true_ite = torch.tensor(
        true_ite_np,
        dtype=torch.float32,
        device=DEVICE,
    )

    true_ate = float(np.mean(true_ite_np))

    # -----------------------------------------------------
    # Build and train IntervalGP-VAE
    # -----------------------------------------------------
    set_seed(GLOBAL_SEED + iteration_id)

    model = build_model(z_train_all)

    model = train_intervalgpvae_model(
        model=model,
        z_train_all=z_train_all,
        t_train_all=t_train_all,
        y_train_all=y_train_all,
        u_train_true_all=u_train_true_all,
    )

    rows = []

    # -----------------------------------------------------
    # Run ablation variants
    # -----------------------------------------------------
    for nominal in NOMINAL_LEVELS:

        # =================================================
        # B0: Point only
        # =================================================
        b0_point = point_ite_from_encoder_mean(
            model=model,
            z=z_test,
        )

        b0_lower = b0_point
        b0_upper = b0_point

        metrics_b0 = compute_all_metrics(
            point=b0_point,
            lower=b0_lower,
            upper=b0_upper,
            true_ite=true_ite,
            nominal=nominal,
        )

        rows.append({
            "iteration_id": iteration_id,
            "method": "B0_point_only",
            "nominal": nominal,
            "true_ate": true_ate,
            **metrics_b0,
        })

        # =================================================
        # B1: Encoder MC latent interval
        # =================================================
        b1 = sample_ite_from_encoder(
            model=model,
            z=z_test,
            n_samples=N_SAMPLES_ITE,
            nominal=nominal,
        )

        metrics_b1 = compute_all_metrics(
            point=b1["mean"],
            lower=b1["lower"],
            upper=b1["upper"],
            true_ite=true_ite,
            nominal=nominal,
        )

        rows.append({
            "iteration_id": iteration_id,
            "method": "B1_encoder_MC_latent",
            "nominal": nominal,
            "true_ate": true_ate,
            **metrics_b1,
        })

        # =================================================
        # B2: Latent-GP smoothed MC interval
        # =================================================
        b2 = sample_ite_from_latent_gp_smoothing(
            model=model,
            z_train=z_train_all,
            z_test=z_test,
            n_samples=N_SAMPLES_ITE,
            nominal=nominal,
        )

        metrics_b2 = compute_all_metrics(
            point=b2["mean"],
            lower=b2["lower"],
            upper=b2["upper"],
            true_ite=true_ite,
            nominal=nominal,
        )

        rows.append({
            "iteration_id": iteration_id,
            "method": "B2_latent_GP_smoothed_MC",
            "nominal": nominal,
            "true_ate": true_ate,
            **metrics_b2,
        })

        # =================================================
        # B3: FullGP GP-ITE posterior
        # =================================================
        b3 = ite_space_intervalgp_head(
            model=model,
            z_train=z_train_all,
            z_test=z_test,
            n_samples=N_SAMPLES_ITE,
            nominal=nominal,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            min_noise=GP_NOISE,
        )

        metrics_b3 = compute_all_metrics(
            point=b3["mean"],
            lower=b3["lower"],
            upper=b3["upper"],
            true_ite=true_ite,
            nominal=nominal,
        )

        rows.append({
            "iteration_id": iteration_id,
            "method": "B3_FullGP_GP_ITE_posterior",
            "nominal": nominal,
            "true_ate": true_ate,
            **metrics_b3,
        })

    df = pd.DataFrame(rows)

    elapsed = time.time() - start_time
    df["time_seconds"] = elapsed

    # -----------------------------------------------------
    # Optional latent recovery diagnostics
    # -----------------------------------------------------
    with torch.no_grad():
        mu_u_test_raw, _, _ = model.vae.get_latent_stats(
            z_test,
            var_name="u",
        )

        mu_u_train, _, _ = model.vae.get_latent_stats(
            z_train_all,
            var_name="u",
        )

        u_test_smooth, u_test_smooth_std = latent_gp_posterior_smoothing(
            z_train=z_train_all,
            mu_u_train=mu_u_train,
            z_test=z_test,
            lengthscale=GP_LENGTHSCALE,
            variance=GP_VARIANCE,
            noise=GP_NOISE,
        )

    _, _, _, raw_corr, raw_rmse = align_recovered_u_to_true_u(
        mu_u_test_raw.squeeze(),
        true_u_test,
    )

    _, _, _, smooth_corr, smooth_rmse = align_recovered_u_to_true_u(
        u_test_smooth.squeeze(),
        true_u_test,
    )

    df["raw_u_corr"] = raw_corr
    df["raw_u_rmse"] = raw_rmse
    df["smooth_u_corr"] = smooth_corr
    df["smooth_u_rmse"] = smooth_rmse

    save_path = os.path.join(
        OUTPUT_DIR,
        f"ablation_no_conformal_iteration_{iteration_id}.csv",
    )

    df.to_csv(save_path, index=False)

    print("\n90% summary for this iteration:")
    print(
        df[df["nominal"] == 0.90][
            [
                "method",
                "PEHE",
                "ATE_error",
                "coverage",
                "mean_width",
                "interval_score",
            ]
        ]
    )

    print(f"\nSaved: {save_path}")
    print(f"Elapsed time: {elapsed:.2f} seconds")

    return df


# =========================================================
# 11. Run all combinations
# =========================================================

def run_all_ablation_experiments():
    all_combinations = list(
        itertools.product(
            proxy_func_sets,
            treatment_func_list,
            outcome_func_list,
        )
    )

    if MAX_ITERATIONS is not None:
        all_combinations = all_combinations[:MAX_ITERATIONS]

    all_results = []

    for iteration_id, (proxy_funcs, treat_func, outcome_func) in enumerate(
        all_combinations,
        start=1,
    ):
        df_one = run_one_combination_ablation(
            iteration_id=iteration_id,
            proxy_funcs=proxy_funcs,
            treat_func=treat_func,
            outcome_func=outcome_func,
        )

        all_results.append(df_one)

        partial_df = pd.concat(all_results, ignore_index=True)

        partial_path = os.path.join(
            OUTPUT_DIR,
            "ablation_no_conformal_partial_results.csv",
        )

        partial_df.to_csv(partial_path, index=False)

        print("\nCurrent partial average 90% summary:")
        print(
            partial_df[partial_df["nominal"] == 0.90]
            .groupby("method", as_index=False)
            .agg(
                PEHE=("PEHE", "mean"),
                ATE_error=("ATE_error", "mean"),
                coverage=("coverage", "mean"),
                mean_width=("mean_width", "mean"),
                interval_score=("interval_score", "mean"),
                raw_u_corr=("raw_u_corr", "mean"),
                smooth_u_corr=("smooth_u_corr", "mean"),
                time_seconds=("time_seconds", "mean"),
            )
        )

    all_results_df = pd.concat(all_results, ignore_index=True)

    full_path = os.path.join(
        OUTPUT_DIR,
        "ablation_no_conformal_all_results.csv",
    )

    all_results_df.to_csv(full_path, index=False)

    summary_df = (
        all_results_df
        .groupby(["method", "nominal"], as_index=False)
        .agg(
            PEHE=("PEHE", "mean"),
            ATE_error=("ATE_error", "mean"),
            coverage=("coverage", "mean"),
            mean_width=("mean_width", "mean"),
            interval_score=("interval_score", "mean"),
            raw_u_corr=("raw_u_corr", "mean"),
            raw_u_rmse=("raw_u_rmse", "mean"),
            smooth_u_corr=("smooth_u_corr", "mean"),
            smooth_u_rmse=("smooth_u_rmse", "mean"),
            time_seconds=("time_seconds", "mean"),
        )
    )

    summary_path = os.path.join(
        OUTPUT_DIR,
        "ablation_no_conformal_summary_mean.csv",
    )

    summary_df.to_csv(summary_path, index=False)

    print("\n" + "=" * 100)
    print("Final average 90% summary")
    print("=" * 100)

    print(
        summary_df[summary_df["nominal"] == 0.90][
            [
                "method",
                "PEHE",
                "ATE_error",
                "coverage",
                "mean_width",
                "interval_score",
                "raw_u_corr",
                "smooth_u_corr",
                "time_seconds",
            ]
        ]
    )

    print("\nSaved full results to:")
    print(full_path)

    print("\nSaved summary table to:")
    print(summary_path)

    plot_metric_bars_at_90(
        summary_df=summary_df,
        output_dir=OUTPUT_DIR,
    )

    plot_coverage_width_tradeoff(
        summary_df=summary_df,
        output_dir=OUTPUT_DIR,
    )

    return all_results_df, summary_df


# =========================================================
# 12. Execute
# =========================================================

all_results_df, summary_df = run_all_ablation_experiments()

summary_df

# End